In [ ]:
import os

from omegaconf import OmegaConf

from app.config import load_config
from infra.db.engine import create_tables, init_engine

os.environ["APP_ENV"] = "prod"
os.environ["APP_ENV"] = "dev"

# Config 불러오기
cfg = load_config()
cfg.db.file = "../test-files/voyager.test.db"  # 노트북용 DB 파일 경로 설정
print(OmegaConf.to_yaml(cfg, resolve=True))

# DB 초기화
print("=== DB 초기화 ===")
init_engine(cfg)
create_tables()
print("✅ DB 엔진 및 테이블 초기화 완료")

In [ ]:
from app.file.file_services import convert_to_file_entry_schema
from core.file_crawler.extractor import convert_path_stat_osxmetadata, walk_files_concurrently
from infra.schemas.file_entry_schema import FileEntrySchema
from utils.paths import get_root_path

project_root = get_root_path()
test_files_dir = project_root / "notebooks" / "test-files"


file_paths = walk_files_concurrently(test_files_dir)
file_entries: list[FileEntrySchema] = []
processed_count: int = 0
for file_path, stat_result, metadata in convert_path_stat_osxmetadata(file_paths):
    try:
        file_entry = convert_to_file_entry_schema(file_path, stat_result, metadata)
        file_entries.append(file_entry)
        processed_count += 1

        if processed_count <= 3:  # 처음 3개만 상세 출력
            print(f"\n✅ 처리 완료: {file_path.name}")
            print(f"   - 크기: {file_entry.size} bytes")
            print(f"   - UTI: {file_entry.uniform_type_identifier}")
            print(f"   - HOME 상대경로: {file_entry.relative_path_from_home}")
            print(f"   - 생성일: {file_entry.creation_date}")

    except Exception as e:
        print(f"❌ 처리 실패: {file_path} - {e}")

file_entries

In [ ]:
len(file_entries)

In [ ]:
from infra.db.engine import get_db_session
from infra.repositories.file_entries import FileEntriesRepository

with get_db_session() as session:
    repo = FileEntriesRepository(session)
    repo.batch_upsert(file_entries)